## Looking at things high level

In [0]:
from pyspark.sql import functions as F

silver = spark.table("dominicks.silver.movement")
silver.printSchema()
print(f"{silver.count():,} rows")
display(silver.limit(10))

In [0]:
# so can delete upc, wstart, ok
# get rid of sale column (inconsistent)
# get rid of source file column as well
# round revenue to nearest cent (nearest hundreth, NOT truncation)
# get rid of is_promo
# eh keep commodity code. i won't use it, but it doesn't hurt
# merge week delta table in. it didn't merge properly last time. write some code to view that first. 


## Didn't merge date labels in properly before, so did that now.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window

path = "/Volumes/dominicks/raw/week_decode/week_decode.json"

# bronze: the raw file as-is
(spark.read.option("multiLine", True).json(path)
    .write.format("delta").mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable("dominicks.bronze.week_decode"))

# silver: typed, no special_event
(spark.table("dominicks.bronze.week_decode").select(
        F.col("week").cast("int").alias("week"),
        F.expr("try_cast(start as date)").alias("week_start"),
        F.expr("try_cast(end as date)").alias("week_end"))
    .write.format("delta").mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable("dominicks.silver.week_decode"))

wk = spark.table("dominicks.silver.week_decode")
wk.printSchema()

In [0]:
# 1. basic sanity: expect 400 rows, 400 distinct weeks, no null dates
display(wk.agg(
    F.count("*").alias("rows"),
    F.countDistinct("week").alias("distinct_weeks"),
    F.min("week").alias("min_week"), F.max("week").alias("max_week"),
    F.sum(F.col("week_start").isNull().cast("int")).alias("null_start"),
    F.sum(F.col("week_end").isNull().cast("int")).alias("null_end")))

# 2. every week is 7 days and starts the day after the previous one ends (expect no rows)
w = Window.orderBy("week")
display(wk.withColumn("len_days", F.datediff("week_end", "week_start") + 1)
          .withColumn("gap", F.datediff("week_start", F.lag("week_end").over(w)))
          .where((F.col("len_days") != 7) | (F.col("gap") != 1)))

# 3. do the weeks in silver all have a calendar entry? (expect no rows)
silver_weeks = spark.table("dominicks.silver.movement").select("week").distinct()
print(silver_weeks.count(), "distinct weeks in silver")
display(silver_weeks.join(wk, "week", "left_anti"))

In [0]:
from pyspark.sql import functions as F

T = "dominicks.silver.movement"
W = "dominicks.silver.week_decode"

# Guards: the calendar must be unique per week, fully dated, and cover every week in silver
wk = spark.table(W)
assert wk.count() == wk.select("week").distinct().count(), "duplicate weeks in calendar"
assert wk.where("week_start IS NULL OR week_end IS NULL").count() == 0, "null dates in calendar"
uncovered = spark.table(T).select("week").distinct().join(wk, "week", "left_anti").count()
assert uncovered == 0, f"{uncovered} weeks in silver have no calendar entry"

n_before = spark.table(T).count()

# Add the columns right after `week` (skipped if they already exist)
if "week_start" not in spark.table(T).columns:
    spark.sql(f"ALTER TABLE {T} ADD COLUMNS (week_start DATE AFTER week, week_end DATE AFTER week_start)")
print(spark.table(T).columns)

In [0]:
spark.sql(f"""
MERGE INTO {T} AS m
USING {W} AS w
ON m.week = w.week
WHEN MATCHED THEN UPDATE SET m.week_start = w.week_start, m.week_end = w.week_end
""")

res = spark.table(T)
assert res.count() == n_before, "row count changed"
nulls = res.where("week_start IS NULL OR week_end IS NULL").count()
assert nulls == 0, f"{nulls:,} rows still have null dates"

print(res.columns)
display(res.select("week", "week_start", "week_end").distinct().orderBy("week").limit(10))

## Now cleaning up useless columns

In [0]:
# so can delete upc, wstart, ok
# get rid of sale column (inconsistent)
# get rid of source file column as well
# round revenue to nearest cent (nearest hundreth, NOT truncation)
# get rid of is_promo
# eh keep commodity code. i won't use it, but it doesn't hurt


In [0]:
from pyspark.sql import functions as F

T = "dominicks.silver.movement"
before = spark.table(T)
n_before = before.count()
rev_before = before.agg(F.sum("revenue")).first()[0]

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {T}
COMMENT 'Weekly store-level scanner movement, typed and validated, with UPC descriptions and week dates'
CLUSTER BY (category, store, week)
AS SELECT
  category, store, week, week_start, week_end,
  move, qty, price, profit, unit_price,
  CAST(ROUND(revenue, 2) AS DECIMAL(18,2)) AS revenue,
  com_code, descrip, size, `case`, nitem
FROM {T}
""")

In [0]:
from pyspark.sql import functions as F

after = spark.table(T)
assert after.count() == n_before, "row count changed"
assert after.where("week_start IS NULL OR week_end IS NULL").count() == 0

rev_after = float(after.agg(F.sum("revenue")).first()[0])
rev_before_f = float(rev_before)
print(rev_before_f, rev_after)
assert abs(rev_after - rev_before_f) / abs(rev_before_f) < 1e-4

display(after.limit(10))

In [0]:
from pyspark.sql import functions as F

T = "dominicks.silver.movement"
cur = spark.table(T)
n_before = cur.count()
rev_before = float(cur.agg(F.sum("revenue")).first()[0])   # old revenue, same math as the new gross_revenue

assert cur.where("qty IS NULL OR qty <= 0").count() == 0, "qty must be positive for the division"
if "ok" in cur.columns:                                     # only exists if the drop hasn't run yet
    bad = cur.where("ok IS NULL OR ok != 1").count()
    assert bad == 0, f"{bad:,} rows have ok != 1"
print(f"{n_before:,} rows, old revenue total {rev_before:,.2f}")

## Adding a few useful columns

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {T}
COMMENT 'Weekly store-level scanner movement, typed and validated, with UPC descriptions and week dates'
CLUSTER BY (category, store, week)
AS SELECT
  category, store, week, week_start, week_end,
  move, qty, price, unit_price,
  CAST(ROUND(price * move / qty, 2) AS DECIMAL(18,2)) AS gross_revenue,
  CAST(ROUND(price * move / qty * profit / 100, 2) AS DECIMAL(18,2)) AS net_revenue,
  profit AS margin_percentage,
  100 - profit AS cost_percentage,
  com_code, descrip, size, `case`, nitem
FROM {T}
""")

after = spark.table(T)
assert after.count() == n_before, "row count changed"
assert after.where("week_start IS NULL OR week_end IS NULL").count() == 0
gross_after = float(after.agg(F.sum("gross_revenue")).first()[0])
assert abs(gross_after - rev_before) / abs(rev_before) < 1e-4, (rev_before, gross_after)

after.printSchema()
display(after.agg(F.min("margin_percentage"), F.max("margin_percentage"),
                  F.sum("gross_revenue"), F.sum("net_revenue")))
display(after.limit(10))

## Some Light EDA because I'm curious

Remember: nitem distinsguishes when it needs to, but also distinguishes when it's not supposed to. good enough for me.

In [0]:
from pyspark.sql import functions as F, Window as W
import matplotlib.pyplot as plt

df = (spark.table("dominicks.silver.movement")
      .withColumn("rev", F.col("gross_revenue").cast("double")))

In [0]:
desc = (df.where("descrip IS NOT NULL")
          .groupBy("nitem")
          .agg(F.min("descrip").alias("descrip")))

In [0]:
counts = (df.groupBy("category")
          .agg(F.countDistinct("nitem").alias("n_nitem"),
               F.countDistinct("descrip", "size").alias("n_descrip_size"),
               F.count("*").alias("rows"))
          .orderBy(F.desc("n_nitem")))
display(counts)

In [0]:
#standard aggregate revenue plots (not accounting for inflation)

ts = (df.groupBy("category", "week_start").agg(F.sum("rev").alias("rev"))
        .orderBy("week_start").toPandas())
cats = sorted(ts["category"].unique())
fig, axes = plt.subplots(7, 4, figsize=(18, 20), sharex=True)
for ax, c in zip(axes.flat, cats):
    d = ts[ts.category == c]
    ax.plot(d["week_start"], d["rev"], lw=0.8)
    ax.set_title(c, fontsize=9)
for ax in axes.flat[len(cats):]:
    ax.axis("off")
fig.suptitle("Weekly gross revenue by category")
plt.tight_layout(); plt.show()

In [0]:
#Dominant items by revenue share per category

desc = (df.where("descrip IS NOT NULL")
          .groupBy("nitem")
          .agg(F.min("descrip").alias("descrip")))

item_rev = df.groupBy("category", "nitem").agg(F.sum("rev").alias("rev"))
w_cat = W.partitionBy("category")
w_cum = w_cat.orderBy(F.desc("rev")).rowsBetween(W.unboundedPreceding, 0)
share = (item_rev
         .withColumn("share_pct", 100 * F.col("rev") / F.sum("rev").over(w_cat))
         .withColumn("rank", F.row_number().over(w_cat.orderBy(F.desc("rev"))))
         .withColumn("cum_share_pct", F.sum("share_pct").over(w_cum)))

top5 = (share.where("rank <= 5").join(desc, "nitem", "left")
        .select("category", "rank", "nitem", "descrip", "rev", "share_pct", "cum_share_pct")
        .orderBy("category", "rank"))
display(top5)

# how concentrated is each category? items needed to reach 80% of revenue
display(share.where("cum_share_pct - share_pct < 80")
        .groupBy("category").agg(F.count("*").alias("items_for_80pct"))
        .join(item_rev.groupBy("category").agg(F.count("*").alias("n_items")), "category")
        .orderBy("category"))

#ok these make sense. even for several decades ago
#huh concentration is kinda cool. didn't think of that one.

In [0]:
#Items with the most price variance per category

wk = (df.groupBy("category", "nitem", "week")
        .agg((F.sum("rev") / F.sum("move")).alias("price"),
             F.sum("rev").alias("rev"))
        .where("price > 0"))
pv = (wk.groupBy("category", "nitem")
        .agg(F.count("*").alias("weeks"),
             F.mean("price").alias("mean_price"),
             (F.stddev("price") / F.mean("price")).alias("price_cv"),
             F.sum("rev").alias("rev"))
        .where("weeks >= 52")
        .withColumn("rank", F.row_number().over(
            W.partitionBy("category").orderBy(F.desc("price_cv")))))

top5 = (pv.where("rank <= 5").join(desc, "nitem", "left")
        .select("category", "rank", "nitem", "descrip", "weeks", "mean_price", "price_cv", "rev")
        .orderBy("category", "rank"))
display(top5)

#not gonna lie i'm not sure why i care about this.

In [0]:
# Price/movement correlation (elasticity proxy)

em = (df.where("move > 0")
        .groupBy("category", "nitem", "week")
        .agg(F.expr("try_divide(sum(rev), sum(move))").alias("price"),
             F.sum("move").alias("units"))
        .where("price > 0 AND units > 0")
        .withColumn("lp", F.log("price")).withColumn("lu", F.log("units")))

el = (em.groupBy("category", "nitem")
        .agg(F.count("*").alias("weeks"),
             F.expr("try_divide(covar_samp(lp, lu), stddev_samp(lp) * stddev_samp(lu))").alias("corr"),
             F.expr("try_divide(covar_samp(lp, lu), var_samp(lp))").alias("elasticity"))
        .where("weeks >= 52 AND corr IS NOT NULL AND elasticity IS NOT NULL")
        .withColumn("rank", F.row_number().over(
            W.partitionBy("category").orderBy("corr"))))

top5 = (el.where("rank <= 5").join(desc, "nitem", "left")
        .select("category", "rank", "nitem", "descrip", "weeks", "corr", "elasticity")
        .orderBy("category", "rank"))
display(top5)

In [0]:
# Revenue for top 5 stores

st = (df.groupBy("store", "week_start").agg(F.sum("rev").alias("rev"))
        .orderBy("week_start").toPandas())
p = st.pivot(index="week_start", columns="store", values="rev")
top = p.sum().nlargest(5).index
fig, ax = plt.subplots(figsize=(14, 6))
p.drop(columns=top).plot(ax=ax, color="lightgray", lw=0.6, legend=False)
p[top].plot(ax=ax, lw=1.4)
ax.set_title("Weekly gross revenue by store (top 5 highlighted)")
ax.set_ylabel("revenue"); plt.show()

#interesting spike.

## Silver to gold delta table

In [0]:
spark.sql("""
ALTER TABLE dominicks.silver.movement
ADD COLUMNS (total_calculated_cost DECIMAL(18,2) AFTER margin_percentage)
""")

spark.sql("""
UPDATE dominicks.silver.movement
SET total_calculated_cost =
  CAST(ROUND(price * move / qty * (100 - margin_percentage) / 100, 2) AS DECIMAL(18,2))
""")

spark.table("dominicks.silver.movement").printSchema()

In [0]:
spark.sql("DROP TABLE IF EXISTS dominicks.gold.store_item_week")

spark.sql("""
CREATE OR REPLACE TABLE dominicks.gold.movement
COMMENT 'Gold weekly store-level scanner movement, one row per silver row'
CLUSTER BY (category, store, week)
AS
SELECT
  category, store, week, week_start, week_end,
  move, qty, price, unit_price,
  gross_revenue, net_revenue, margin_percentage,
  total_calculated_cost, cost_percentage,
  com_code, descrip, size, `case`, nitem
FROM dominicks.silver.movement
""")

In [0]:
s = spark.table("dominicks.silver.movement")
g = spark.table("dominicks.gold.movement")
print(s.count(), g.count())
print(s.schema == g.schema)
g.printSchema()

In [0]:
display(spark.table("dominicks.gold.movement")
        .orderBy("category", "store", "nitem", "week").limit(10))